# Medical Appointment No-Show Analysis

## Project Overview
Analyze appointment data to identify patterns in no-shows across timing, patient history, clinic type, and demographics.

## Learning Objectives
- Calculate no-show rates across patient and scheduling dimensions
- Identify high-risk appointment types and patient profiles for no-shows
- Analyze the impact of lead time, reminders, and scheduling patterns
- Quantify the revenue and capacity impact of no-shows

## Problem Statement
Clinics lose significant revenue and capacity to no-show appointments. Understanding which patients and scheduling patterns predict no-shows enables targeted interventions like reminders, overbooking, and outreach.

## Why This Project Matters
No-shows waste provider time, delay care for other patients, and cost the US healthcare system an estimated $150 billion annually. Even small reductions in no-show rates yield large operational gains.

## Dataset Overview
Synthetic appointment dataset: ~10,000 appointments with patient demographics, scheduling details, clinic info, and show/no-show status.

## Dataset Source and License Notes
- Synthetic data inspired by Kaggle Medical Appointment No-Shows dataset
- No license restrictions

## Environment Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import matplotlib
matplotlib.use('Agg')

## Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid', palette='Set2')
np.random.seed(42)
print('Imports OK')

## Configuration / Constants

In [ ]:
import os
SAVE_DIR = os.path.dirname(os.path.abspath('__file__'))
print(f'Save dir: {SAVE_DIR}')

## Dataset Download or Loading

In [ ]:
np.random.seed(42)
n = 10000
age = np.clip(np.random.normal(45, 18, n).astype(int), 5, 90)
gender = np.random.choice(['Male', 'Female'], n, p=[0.42, 0.58])
insurance = np.random.choice(['Medicare', 'Medicaid', 'Private', 'Self-Pay'], n, p=[0.28, 0.20, 0.38, 0.14])
clinic = np.random.choice(['Primary Care', 'Specialty', 'Dental', 'Mental Health', 'Urgent Care'],
                            n, p=[0.30, 0.25, 0.18, 0.15, 0.12])
appt_day = np.random.choice(['Monday','Tuesday','Wednesday','Thursday','Friday'], n, p=[0.22, 0.20, 0.20, 0.20, 0.18])
lead_days = np.clip(np.random.lognormal(2.0, 0.8, n), 0, 90).round(0).astype(int)
prior_noshow = np.clip(np.random.poisson(0.5, n), 0, 10)
sms_reminder = (np.random.random(n) < 0.65).astype(int)
chronic_condition = (np.random.random(n) < 0.35).astype(int)
time_slot = np.random.choice(['Morning', 'Afternoon', 'Late Afternoon'], n, p=[0.40, 0.35, 0.25])

# No-show logic
noshow_prob = 0.08 +     0.06 * (lead_days > 14) + 0.04 * (lead_days > 30) +     0.05 * (prior_noshow >= 2) + 0.03 * (prior_noshow >= 1) +     0.04 * (insurance == 'Medicaid') + 0.03 * (insurance == 'Self-Pay') +     -0.05 * sms_reminder +     0.03 * (clinic == 'Mental Health') +     0.02 * (age < 30) +     np.random.normal(0, 0.02, n)
noshow_prob = np.clip(noshow_prob, 0.02, 0.45)
noshow = (np.random.random(n) < noshow_prob).astype(int)

df = pd.DataFrame({
    'AppointmentID': [f'A{i:06d}' for i in range(n)],
    'Age': age, 'Gender': gender, 'Insurance': insurance,
    'Clinic': clinic, 'AppointmentDay': appt_day,
    'LeadDays': lead_days, 'PriorNoShows': prior_noshow,
    'SMSReminder': sms_reminder, 'ChronicCondition': chronic_condition,
    'TimeSlot': time_slot, 'NoShow': noshow,
})
df['AgeBand'] = pd.cut(df['Age'], bins=[0, 18, 35, 50, 65, 100], labels=['<18', '18-35', '35-50', '50-65', '65+'])
df['LeadBand'] = pd.cut(df['LeadDays'], bins=[-1, 1, 7, 14, 30, 999],
                          labels=['Same/Next Day', '2-7 days', '1-2 weeks', '2-4 weeks', '4+ weeks'])

print(f'Dataset: {df.shape}')
print(f'Overall no-show rate: {df["NoShow"].mean():.1%}')
df.head()

## Data Validation Checks

In [ ]:
print(f'Missing values: {df.isnull().sum().sum()}')
print(f'\nClinic distribution:\n{df["Clinic"].value_counts()}')
print(f'\nNo-show counts: {df["NoShow"].value_counts().to_dict()}')

## Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

df.groupby('Clinic')['NoShow'].mean().sort_values().plot.barh(ax=axes[0,0], edgecolor='black', color='coral')
axes[0,0].set_title('No-Show Rate by Clinic Type')

df.groupby('LeadBand')['NoShow'].mean().plot.bar(ax=axes[0,1], edgecolor='black', color='steelblue')
axes[0,1].set_title('No-Show Rate by Lead Time')
axes[0,1].tick_params(axis='x', rotation=30)

df.groupby('AgeBand')['NoShow'].mean().plot.bar(ax=axes[1,0], edgecolor='black', color='teal')
axes[1,0].set_title('No-Show Rate by Age Band')
axes[1,0].tick_params(axis='x', rotation=0)

sms_effect = df.groupby('SMSReminder')['NoShow'].mean()
sms_effect.index = ['No SMS', 'SMS Sent']
sms_effect.plot.bar(ax=axes[1,1], edgecolor='black', color='goldenrod')
axes[1,1].set_title('No-Show Rate: SMS Reminder Effect')
axes[1,1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'eda_overview.png'), dpi=100, bbox_inches='tight')
plt.show()

## Prior No-Show History Impact

In [ ]:
prior_rate = df.groupby('PriorNoShows').agg(
    appointments=('AppointmentID', 'count'),
    noshow_rate=('NoShow', 'mean'),
).round(3)
print('No-Show Rate by Prior No-Show Count:')
print(prior_rate)

fig, ax = plt.subplots(figsize=(10, 5))
prior_rate['noshow_rate'].plot(ax=ax, marker='o', color='coral')
ax.set_title('No-Show Rate by Number of Prior No-Shows')
ax.set_xlabel('Prior No-Shows')
ax.set_ylabel('No-Show Rate')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'prior_noshow.png'), dpi=100, bbox_inches='tight')
plt.show()

## Insurance and Scheduling Patterns

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ins_rate = df.groupby('Insurance')['NoShow'].mean().sort_values()
ins_rate.plot.barh(ax=axes[0], edgecolor='black', color='steelblue')
axes[0].set_title('No-Show Rate by Insurance')

slot_rate = df.groupby('TimeSlot')['NoShow'].mean()
slot_rate.plot.bar(ax=axes[1], edgecolor='black', color='teal')
axes[1].set_title('No-Show Rate by Time Slot')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'insurance_scheduling.png'), dpi=100, bbox_inches='tight')
plt.show()

## SMS Reminder Effectiveness by Segment

In [ ]:
sms_by_clinic = df.groupby(['Clinic', 'SMSReminder'])['NoShow'].mean().unstack()
sms_by_clinic.columns = ['No SMS', 'SMS Sent']
sms_by_clinic['Reduction'] = ((sms_by_clinic['No SMS'] - sms_by_clinic['SMS Sent']) / sms_by_clinic['No SMS'] * 100).round(1)
print('SMS Reminder Effectiveness by Clinic:')
print(sms_by_clinic.round(3))

## Revenue Impact Estimation

In [ ]:
avg_appt_value = {'Primary Care': 150, 'Specialty': 300, 'Dental': 200, 'Mental Health': 175, 'Urgent Care': 250}
noshows = df[df['NoShow'] == 1]
noshows_by_clinic = noshows.groupby('Clinic').size()
revenue_loss = pd.Series({c: noshows_by_clinic.get(c, 0) * avg_appt_value[c] for c in avg_appt_value})
print('Estimated Revenue Loss from No-Shows:')
print(revenue_loss.sort_values(ascending=False))
print(f'\nTotal estimated loss: ${revenue_loss.sum():,.0f}')

## Interpretation and Business Insight
- **Lead time** is the strongest scheduling factor — appointments booked 4+ weeks out have much higher no-show rates
- **Prior no-show history** is the best patient-level predictor — past behavior predicts future behavior
- **SMS reminders** reduce no-shows by ~25-35% — they should be universal, not optional
- **Mental Health** clinics have higher no-show rates — stigma, ambivalence, and access barriers play a role
- **Young adults (18-35)** and **Medicaid/Self-Pay** patients no-show more often — targeted outreach matters
- **Revenue impact** is substantial — even 5% reduction in no-shows recovers significant revenue

## Limitations
- Synthetic data with simplified no-show logic
- No transportation, weather, or competing appointment data
- No patient preference or engagement data
- No call/confirmation tracking beyond SMS
- No distinction between cancellation and true no-show

## How to Improve This Project
- Add predictive no-show models for proactive double-booking
- Include transportation and distance-to-clinic data
- Track multiple reminder modalities (SMS, call, email, app)
- Add weather and day-of correlation analysis
- Model optimal overbooking rates by clinic and day

## Production Considerations
- Real-time no-show risk scoring at scheduling time
- Automated reminder sequences (SMS + call for high-risk)
- Dynamic overbooking algorithms by clinic and slot
- Monthly no-show dashboards for clinic managers

## Common Mistakes
- Treating no-shows as purely a patient problem (access barriers matter)
- Not leveraging prior no-show history for risk scoring
- Uniform overbooking instead of risk-stratified scheduling
- Penalizing patients instead of addressing root causes

## Mini Challenge / Exercises
1. Calculate the no-show rate for patients with 3+ prior no-shows AND no SMS reminder.
2. If SMS reminders were sent to ALL appointments, estimate the total no-show reduction.
3. Which Clinic × Insurance combination has the highest no-show rate?
4. Design a simple 3-tier risk score using lead days, prior no-shows, and SMS status.

## Final Summary / Key Takeaways
- No-show analysis reveals actionable patterns for scheduling optimization
- Prior history, lead time, and reminder status are the strongest predictors
- SMS reminders are highly effective and should be standard for all appointments
- Risk-stratified scheduling (overbooking high-risk slots) recovers lost capacity
- Addressing access barriers for underserved populations reduces no-shows and improves equity